In [33]:
import pandas as pd
import yaml
import subprocess

In [34]:
REGENERATE_RULES = False 
RULES_YML_PATH = "data/asca_rules.yml"
RULES_CSV_PATH = "data/asca_rules.csv"
DF_COLS = ["section_index", "section_name", "rule_id", "step_index", "input", "output", "env", "exception"]

if REGENERATE_RULES:
    cmd = ["uv", "run", "compile_rules", "--compiler-config", "../config/pipeline/compiler_config.yml", "--compiled-out", RULES_YML_PATH]

    output = subprocess.run(
        cmd, capture_output=True, timeout=60, text=True, check=True,
    )
    output.check_returncode()

    with open(RULES_YML_PATH, "r") as f:
        rules_doc = yaml.load(f, Loader=yaml.SafeLoader)

    rules = []
    for s in rules_doc["sections"]:
        for r in s["rules"]:
            if "steps" not in r:
                rules.append({"section_index": s["index"], "section_name": s["section"], "step_index": 0, **r})
            else:
                for i,step in enumerate(r["steps"]):
                    rules.append({"section_index": s["index"], "section_name": s["section"], "step_index": i, "rule_id": r["rule_id"], **step})


    rules_df = pd.DataFrame(rules)[DF_COLS]
    rules_df.to_csv(RULES_CSV_PATH, index=False)
else:
    rules_df = pd.read_csv(RULES_CSV_PATH)

In [35]:
rules_df.head(20)

,section_index,section_name,rule_id,step_index,input,output,env,exception
0,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,dz ʃ tʃ,ʒ F F,NaN,NaN
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,0,dʒ,tʃ,NaN,NaN
2,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,1,tʃ,ʃ,NaN,NaN
3,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-t,0,t,∅,_s#,NaN
4,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-ɬ,0,ɬ,l,NaN,NaN
5,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-f,0,f,p,NaN,NaN
6,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-aː,0,"{a,a:[+long]}","{e,e:[+long]}","_{ʕ,q}$",NaN
7,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,q ʕ,ʔ h,NaN,NaN
8,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-a,0,a,o,"#Cw_{dl,l,F}",NaN
9,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-w,0,w,∅,#C_V,"_{i,i:[+long]}"


In [37]:
segments = pd.concat([rules_df["input"], rules_df["output"]])
contexts = pd.concat([rules_df["env"], rules_df["exception"]]).dropna()

print(segments.value_counts())
print(contexts.value_counts())

∅                                                                       1452
s                                                                        406
h                                                                        381
e                                                                        335
a                                                                        321
                                                                        ... 
N Q:[+spread,+voice] Q͡ɢ                                                   1
ʕm                                                                         1
N Q:[+spread,+voice] {ǀ:[+spread,+voice]͡q,ǀ:[+spread,+voice]͡ɢ} Q͡q       1
di                                                                         1
{kǁ:[+cg],kǁx:[+cg]}                                                       1
Name: count, Length: 4379, dtype: int64
_#                                               483
#_                                               358
V_V    